# 14 — V3-E2: auditoria do cache de embeddings MiniLM

Este notebook verifica a evidência operacional produzida no V3-E2. Ele é
somente leitura: não recalcula embeddings, não treina classificadores, não
seleciona threshold e não acessa `test`, `stress` ou `monitor`.

O objetivo é confirmar que o cache local corresponde integralmente às chaves
de desenvolvimento autorizadas e que seus arquivos permanecem íntegros. As
saídas são exclusivamente agregadas; nenhuma narrativa ou embedding individual
é publicado.


## Contrato científico congelado

O V3-E0 fixou a classe crítica `debt_credit_management`, MiniLM congelado com
LinearSVC como abordagem principal, TF-IDF word/char com LinearSVC como
controle, todos os positivos críticos únicos, razões negativas 1:5, 1:10 e
1:15, e validação temporal crescente com prevalência natural e precisão mínima
de 0,40.

O E2 implementa apenas a representação MiniLM congelada. Comparações
preditivas entre representações, razões e folds pertencem ao V3-E3 e ainda não
existem neste ponto.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from consumer_complaint_intelligence.v3.reporting import (
    audit_v3_e2_cache,
    render_v3_e2_audit,
)


## Auditoria integral do cache

A chamada abaixo valida o manifesto e o hash do protocolo, os hashes e tamanhos
dos 19 pares de shards, formato e dimensão dos arrays, esquema das chaves,
unicidade dos grupos e correspondência exata com `train` e `validation` no
cache científico. Também calcula somente diagnósticos agregados das normas L2.


In [2]:
audit = audit_v3_e2_cache(PROJECT_ROOT)
print(render_v3_e2_audit(audit, 'pt-BR'))

V3-E2 — AUDITORIA DO CACHE DE EMBEDDINGS

status: VALID_COMPLETE_DEVELOPMENT_CACHE
model: sentence-transformers/all-MiniLM-L6-v2
revision: bc57282bc374d33e0d6c4de27f12dc1c2a87f37a
shape: 591,532 x 384
dtype: float16

INTEGRIDADE
  shards: 19
  unique_groups: 591,532
  train_rows: 345,552
  validation_rows: 245,980
  missing_keys: 0
  extra_keys: 0
  raw_narratives_persisted: False
  forbidden_partitions_accessed: False
  cache_size_mib: 444.94
  ordered_key_sha256: 87F159E3BD1461E82F03FAF7D5AAAA966B0E7F12E5D45B4827476012CAF25AB4
  manifest_sha256: 09C67DF7B0EF6BC25FCFFFAB3E4EEEFB37E3EE75E2F0B07052F44AB9CEA235E5

EXECUÇÃO
  elapsed_seconds: 717.5706
  rows_per_second: 824.35
  batch_size: 128
  device: NVIDIA GeForce RTX 3050 Laptop GPU
  peak_reserved_mib: 488.00

NORMAS L2 DOS EMBEDDINGS (DIAGNÓSTICO)
  train: n=345,552 | min=0.9995 | p01=0.9996 | median=1.0000 | mean=1.0000 | p99=1.0004 | max=1.0005 | zeros=0
  validation: n=245,980 | min=0.9995 | p01=0.9996 | median=1.0000 | mean=1.

## Como interpretar

Um status válido demonstra integridade, cobertura e viabilidade operacional do
cache. As normas L2 ajudam a detectar vetores nulos, não finitos ou deriva
grosseira de escala; elas não medem separabilidade nem qualidade preditiva.

As normas observadas ficam próximas de 1 porque o próprio checkpoint fixado
contém um módulo interno `2_Normalize`. O parâmetro congelado
`normalize_embeddings=False` significa que a chamada `encode` não solicita
uma normalização adicional; ele não remove o módulo do checkpoint.

Portanto, este notebook não permite afirmar que MiniLM é melhor ou pior que
TF-IDF. Essa pergunta começa no V3-E3, com os folds rolling-origin, a amostragem
congelada e a calibração temporal já aprovadas.
